In [1]:
import logging
from transformers import logging as tf_logging

tf_logging.set_verbosity_error()

In [2]:
import sys

In [3]:
import asyncio
import json
import os
import time
from pathlib import Path

import dotenv
from tqdm import tqdm

from financial_qa.chunkers.table_aware_recursive import TableAwareRecursiveChunker
from financial_qa.embedders import GigaEmbedder
from financial_qa.rag import RAG
from financial_qa.agent.agent_loop import OpenRouterAgentLoop
from financial_qa.agent.gigachat_agent_loop import GigaChatAgentLoop
from financial_qa.evaluation import evaluate_async, load_jsonl

In [4]:
dotenv.load_dotenv('.env')
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')

GIGACHAT_CREDENTIALS = os.getenv('GIGACHAT_CREDENTIALS')
if not GIGACHAT_CREDENTIALS:
    raise ValueError('GIGACHAT_CREDENTIALS is required')

DATASET_FILE = 'dataset.jsonl'
DATASET_SPLIT = None
MAX_QUESTIONS = None

RAG_DB = 'tarec_chunk_giga_embeddings_r'
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 10
EMBED_MODEL = 'EmbeddingsGigaR'
GIGACHAT_SCOPE = 'GIGACHAT_API_PERS'

GEN_MODEL = "GigaChat-2-Pro"
QUERY_CONCURRENCY = 1

JUDGE_MODEL = 'google/gemini-2.0-flash-lite-001'
JUDGE_PROCESSES = 100  # None => one process per question
USE_GIGACHAT_JUDGE = True

In [5]:
all_records = load_jsonl(DATASET_FILE)

In [6]:
len(all_records)

449

In [7]:
all_records = load_jsonl(DATASET_FILE)

all_records = [
    all_records[r]
    for r in all_records
    if DATASET_SPLIT is None or all_records[r].get('split') == DATASET_SPLIT
]

seen = set()
records = []
cnt = 0
for r in all_records[50:100]:
    if r['question_id'] not in seen:
        records.append(r)
        seen.add(r['question_id'])
        cnt += 1
    else:
        print('huy')
print(cnt)

if MAX_QUESTIONS:
    records = records[:MAX_QUESTIONS]

golden = {r['question_id']: r for r in records}
print(f'Loaded {len(records)} records (split={DATASET_SPLIT!r})')
print('Sample:', json.dumps(records[0], ensure_ascii=False, indent=2))

50
Loaded 50 records (split=None)
Sample: {
  "question_id": "q_462f7b5a3abcabad",
  "question": "Начиная с какой даты Совкомбанк применяет новые стандарты учётной политики в обобщённой промежуточной консолидированной финансовой отчётности за шесть месяцев, завершившихся 30 июня 2023 года?",
  "split": "test",
  "gold_evidence": [
    {
      "doc_id": "sovkom_2023_6m",
      "pages": [
        10
      ]
    }
  ],
  "gold_answer": "С 1 января 2023 года."
}


In [8]:
# chunker = SlidingWindowChunker(chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
chunker = TableAwareRecursiveChunker(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
embedder = GigaEmbedder(
    credentials=GIGACHAT_CREDENTIALS,
    model=EMBED_MODEL,
    scope=GIGACHAT_SCOPE,
)
rag = RAG(
    chunker=chunker,
    embedder=embedder,
    data_dir='data/parsed',
    store_dir='indexes',
    name=RAG_DB,
    top_k=TOP_K,
)

store_dir = Path('indexes') / RAG_DB
has_index = store_dir.exists() and any(store_dir.glob('*.npz'))
if not has_index:
    print('No index found; running precalc...')
    rag.precalc()
else:
    print(f'Using existing index at {store_dir}')

loop = GigaChatAgentLoop(
    rag=rag,
    model=GEN_MODEL,
    credentials=GIGACHAT_CREDENTIALS,
    scope=GIGACHAT_SCOPE,
)

Using existing index at indexes/tarec_chunk_giga_embeddings_r


In [9]:
async def run_queries(records):
    predictions = {}
    errors = []
    timings = []
    semaphore = asyncio.Semaphore(QUERY_CONCURRENCY)

    async def _query_one(rec):
        start = time.perf_counter()
        try:
            answer, confidence = await loop.aquery(rec['question'])
            error = None
        except Exception as e:
            answer = ''
            confidence = None
            error = str(e)
        elapsed = time.perf_counter() - start
        return {
            'question_id': rec['question_id'],
            'question': rec['question'],
            'answer': answer,
            'evidence': [],
            'confidence': confidence,
            'error': error,
            'elapsed_s': elapsed,
        }

    async def _bound(rec):
        async with semaphore:
            return await _query_one(rec)

    tasks = {asyncio.create_task(_bound(rec)): rec for rec in records}
    progress = tqdm(total=len(tasks), desc='Querying agent', unit='question')
    for task in asyncio.as_completed(tasks):
        result = await task
        predictions[result['question_id']] = result
        if result['error']:
            errors.append(result)
        timings.append(result['elapsed_s'])
        progress.update(1)
        progress.set_postfix(
            errors=len(errors),
            avg_s=f"{sum(timings)/len(timings):.2f}",
            last_conf=result['confidence'],
        )
    progress.close()
    return predictions, errors

predicted, query_errors = await run_queries(records)
print(f'Done: {len(predicted)} answers, {len(query_errors)} errors')

Querying agent:   0%|          | 0/50 [00:00<?, ?question/s]

[GigaChat] 429 rate limit (attempt 1/8), sleeping 16s…


Querying agent:  68%|██████▊   | 34/50 [02:59<01:46,  6.66s/question, avg_s=5.27, errors=0, last_conf=93.1]

[GigaChat] 429 rate limit (attempt 1/8), sleeping 19s…


Querying agent:  72%|███████▏  | 36/50 [03:27<02:12,  9.43s/question, avg_s=5.76, errors=0, last_conf=95.8]

[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 11s…


Querying agent: 100%|██████████| 50/50 [04:50<00:00,  5.81s/question, avg_s=5.81, errors=0, last_conf=92.8]

Done: 50 answers, 0 errors


In [10]:
query_errors

[]

In [11]:
result = await evaluate_async(
    golden=golden,
    predicted=predicted,
    model=JUDGE_MODEL,
    detailed_result=True,
    include_evidence=False,
    use_processes=True,
    max_workers=JUDGE_PROCESSES,
    progress_desc='LLM-as-judge',
)

LLM-as-judge: 100%|██████████| 50/50 [00:01<00:00, 28.90question/s, accuracy=52.00%, correct=26, errors=0]


In [12]:
result['correct'] / result['total']

0.52

In [13]:
for res in result['results'][0:10]:
    print(res)
    print('-' * 75)

{'question_id': 'q_462f7b5a3abcabad', 'question': 'Начиная с какой даты Совкомбанк применяет новые стандарты учётной политики в обобщённой промежуточной консолидированной финансовой отчётности за шесть месяцев, завершившихся 30 июня 2023 года?', 'gold_answer': 'С 1 января 2023 года.', 'predicted_answer': 'Совкомбанк применяет новые стандарты учётной политики в обобщённой промежуточной консолидированной финансовой отчётности за шесть месяцев, завершившихся 30 июня 2023 года, начиная с 1 января 2023 года.', 'judge_score': 1, 'judge_reasoning': 'Оба ответа содержат одну и ту же информацию, хотя предсказанный ответ более развернутый.'}
---------------------------------------------------------------------------
{'question_id': 'q_49b8dba88ad5bf33', 'question': 'Каково итоговое непризнанное изменение в справедливой стоимости финансовых активов и обязательств Совкомбанка по состоянию на 30 июня 2024 года?', 'gold_answer': '(34 863) млн руб.', 'predicted_answer': 'Итоговое непризнанное изменен

In [14]:
import shutil
shutil.make_archive("logs", "zip", "logs")

'/Users/kitlix/CProjects/hse/ai360_proj_may_2026/ai360-financial-qa/logs.zip'